# Section A — Schema Design
Explore the dataset, justify normalisation choices, and generate the ER diagram.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

RAW = Path('../data/raw')
OCT = Path('../data/2019-Oct.csv')
NOV = Path('../data/2019-Nov.csv')

## A1 — Dataset Exploration

In [ ]:
sample = pd.read_csv(OCT, nrows=500_000)
print('Shape:', sample.shape)
print('\nDtypes:')
print(sample.dtypes)
print('\nNull counts:')
print(sample.isnull().sum())
print('\nEvent types:', sample['event_type'].value_counts().to_dict())
print('\nSample rows:')
sample.head(3)

In [ ]:
# Null rate per column
null_pct = (sample.isnull().sum() / len(sample) * 100).round(2)
print('Null % per column:')
print(null_pct)

In [ ]:
# Cardinality check
for col in ['event_type','product_id','category_id','brand','user_id','user_session']:
    print(f'{col}: {sample[col].nunique():,} unique values')

## A1 — Schema Design Rationale

**Normal Form chosen: 3NF**

The raw CSV has a single flat row per event. Functional dependencies present:
- `product_id → category_id, category_code, brand, price`  
- `category_id → category_code`

These transitive dependencies violate 3NF in a flat table, so we decompose into:

| Table | Role | Key |
|---|---|---|
| `dim_users` | User dimension | `user_id` PK |
| `dim_categories` | Category dimension | `category_id` PK |
| `dim_products` | Product dimension | `product_id` PK, FK → `dim_categories` |
| `fact_events` | Event fact table | `event_id` PK, FK → products & users |
| `pipeline_runs` | Audit/logging | `run_id` PK |

**Nullable columns:**
- `category_code`: ~33% null in sample — product may belong to an uncoded category
- `brand`: ~14% null — brand not always recorded

**Index strategy** (see ddl.sql for full list):
- `fact_events(product_id)` — JOIN to dim_products for funnel/brand queries
- `fact_events(event_type)` — WHERE filter in every analytical query
- `fact_events(user_session)` — GROUP BY for session aggregation
- `fact_events(user_id, source_month)` — Oct-only / Nov-only user queries
- `fact_events(event_time)` — hourly distribution query

## ER Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14); ax.set_ylim(0, 9); ax.axis('off')
ax.set_facecolor('#f8f9fa'); fig.patch.set_facecolor('#f8f9fa')

def draw_table(ax, x, y, title, cols, w=3.2, row_h=0.38):
    h = row_h * (len(cols) + 1)
    ax.add_patch(mpatches.FancyBboxPatch((x, y-h), w, h,
        boxstyle='round,pad=0.05', fc='#2c3e50', ec='#1a252f', lw=1.5))
    ax.text(x + w/2, y - row_h/2, title, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    for i, (col, dtype, note) in enumerate(cols):
        bg = '#ecf0f1' if i % 2 == 0 else '#dfe6e9'
        ax.add_patch(mpatches.FancyBboxPatch((x, y - row_h*(i+2)), w, row_h,
            boxstyle='square,pad=0', fc=bg, ec='#bdc3c7', lw=0.5))
        ax.text(x+0.1, y - row_h*(i+1.5), f'{col}  {dtype}', va='center',
                fontsize=7, color='#2c3e50')
        if note:
            ax.text(x+w-0.1, y - row_h*(i+1.5), note, va='center', ha='right',
                    fontsize=6.5, color='#e74c3c', fontweight='bold')

draw_table(ax, 0.3, 8.5, 'dim_users', [
    ('user_id', 'BIGINT', 'PK'),
])

draw_table(ax, 0.3, 5.5, 'dim_categories', [
    ('category_id',   'BIGINT',       'PK'),
    ('category_code', 'VARCHAR(255)', 'NULL'),
])

draw_table(ax, 4.5, 5.5, 'dim_products', [
    ('product_id',    'BIGINT',       'PK'),
    ('category_id',   'BIGINT',       'FK'),
    ('category_code', 'VARCHAR(255)', 'NULL'),
    ('brand',         'VARCHAR(100)', 'NULL'),
    ('price',         'NUMERIC(10,2)','NN'),
])

draw_table(ax, 8.5, 8.8, 'fact_events', [
    ('event_id',     'BIGSERIAL',     'PK'),
    ('event_time',   'TIMESTAMP',     'NN'),
    ('event_type',   'VARCHAR(20)',   'NN'),
    ('product_id',   'BIGINT',        'FK'),
    ('user_id',      'BIGINT',        'FK'),
    ('user_session', 'VARCHAR(64)',   'NN'),
    ('price',        'NUMERIC(10,2)', 'NN'),
    ('source_month', 'CHAR(7)',       'NN'),
])

draw_table(ax, 4.5, 2.2, 'pipeline_runs', [
    ('run_id',          'SERIAL',      'PK'),
    ('source_file',     'VARCHAR',     'NN'),
    ('started_at',      'TIMESTAMP',   'NN'),
    ('finished_at',     'TIMESTAMP',   'NULL'),
    ('rows_extracted',  'INTEGER',     ''),
    ('rows_loaded',     'INTEGER',     ''),
    ('status',          'VARCHAR(20)', ''),
])

# Relationship arrows
arrow = dict(arrowstyle='->', color='#e74c3c', lw=1.5)
# fact_events.product_id → dim_products
ax.annotate('', xy=(7.7, 4.5), xytext=(8.5, 5.5), arrowprops=arrow)
# fact_events.user_id → dim_users
ax.annotate('', xy=(3.5, 7.8), xytext=(8.5, 7.8), arrowprops=arrow)
# dim_products.category_id → dim_categories
ax.annotate('', xy=(3.5, 4.5), xytext=(4.5, 4.5), arrowprops=arrow)

ax.text(7, 9.1, 'E-Commerce Event Schema — 3NF', fontsize=13,
        fontweight='bold', ha='center', color='#2c3e50')
ax.text(13.8, 0.1, 'PK=Primary Key  FK=Foreign Key  NN=Not Null  NULL=Nullable',
        fontsize=7, ha='right', color='#7f8c8d')

plt.tight_layout()
plt.savefig('../schema/er_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print('ER diagram saved to schema/er_diagram.png')

## A3 — Schema Justification

**Why 3NF over alternatives?**  
A fully denormalised single table (1NF) would store `category_code` and `brand` redundantly for every event — wasting storage and making category-level updates inconsistent. BCNF would offer no additional benefit here since all non-trivial functional dependencies are already resolved at 3NF. The chosen decomposition eliminates the transitive dependency `product_id → category_id → category_code` by separating `dim_categories`.

**Trade-offs made:**  
- `price` is stored in both `dim_products` (latest known price) and `fact_events` (price at event time). This controlled redundancy is intentional — product prices change over time and the fact table must record the price at the moment of the event for accurate revenue calculations.
- `category_code` is kept in `dim_products` as a denormalised convenience column to avoid an extra JOIN in the most common funnel query (Q1). This is a deliberate 3NF relaxation for query performance.

**Index justification:**  
Every analytical query in Section C filters or groups on `event_type`, `product_id`, `user_session`, `user_id + source_month`, or `event_time`. Without indexes, each query would require a full sequential scan of the 10M+ row fact table. The composite index on `(user_id, source_month)` directly supports Q4 (Oct-only users) without a full scan. The `event_time` index enables the hourly distribution query (Q5) to use an index scan with `EXTRACT(HOUR FROM event_time)`.